# Momentum Timing: Forecast Signals, Exposure Caps, and the Role of Leverage

Full study sample: **January 2005–April 2026 (256 return months)**. The central comparison separates timing information from leverage and volatility matching. Predictor information at month-end *t* determines the MOM factor payoff in month *t+1*. The DM-style model is additionally trained using January 1966 onward, while the VIX-composite begins in January 1995. Historical sentiment is examined separately through December 2023. Risk is reported using levered factor payoffs **before financing and trading costs**; an illustrative constant 1.6× long–short momentum exposure is not equivalent to a fully specified funded portfolio.

The historical 2005–2023 results are retained in the notebook for comparison, followed by the primary 2005–2026 calculations. For local replication, edit only BASE in the first code cell. Raw files must be directly in that folder; the notebook creates all output folders.

In [ ]:
import os, warnings, math, glob
warnings.filterwarnings("ignore")
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# =============================================================
# CHANGE ONLY THIS LINE WHEN RUNNING LOCALLY
# All raw .csv/.xlsx inputs are directly in BASE (no subfolders required).
# =============================================================
BASE = r"/mnt/data"

RAW = os.path.join(BASE, "raw_data") if os.path.isdir(os.path.join(BASE, "raw_data")) else BASE
OUT = os.path.join(BASE, "momentum_crash_outputs_revised")
FIG = os.path.join(OUT, "figures")
TAB = os.path.join(OUT, "tables")
CLEAN = os.path.join(OUT, "clean_data")
for d in [OUT, FIG, TAB, CLEAN]:
    os.makedirs(d, exist_ok=True)
BASE, RAW, OUT

## 1. Helper functions

In [ ]:
def month_end_from_yyyymm(x):
    x = str(int(x))
    return pd.to_datetime(x, format="%Y%m") + pd.offsets.MonthEnd(0)


def read_ff_monthly(path):
    df = pd.read_csv(path)
    df = df[pd.to_numeric(df["Date"], errors="coerce").notna()].copy()
    df["Date"] = df["Date"].astype(int).apply(month_end_from_yyyymm)
    for c in df.columns:
        if c != "Date":
            df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0
    return df


def read_ff_daily(path):
    df = pd.read_csv(path)
    df = df[pd.to_numeric(df["Date"], errors="coerce").notna()].copy()
    df["Date"] = pd.to_datetime(df["Date"].astype(int).astype(str), format="%Y%m%d")
    for c in df.columns:
        if c != "Date":
            df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0
    return df



def find_input_file(candidates, required=True):
    """Find an input file in RAW or BASE. Candidates may contain exact names or glob patterns."""
    search_dirs = [RAW, BASE]
    for d in search_dirs:
        for cand in candidates:
            matches = glob.glob(os.path.join(d, cand))
            if matches:
                return sorted(matches)[0]
    if required:
        raise FileNotFoundError(f"Could not find any of: {candidates} in {search_dirs}")
    return None


def read_optional_ff3_monthly(path):
    """Optional Fama-French 3-factor monthly file. If supplied, it gives pre-1963 market returns."""
    if path is None:
        return None
    df = read_ff_monthly(path).rename(columns={"Mkt-RF": "Mkt_RF"})
    return df


def nw_reg(y, X, lags=6):
    data = pd.concat([y, X], axis=1).dropna()
    yy = data.iloc[:, 0]
    XX = sm.add_constant(data.iloc[:, 1:])
    model = sm.OLS(yy, XX).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return model


def stars(t):
    at = abs(t)
    return "***" if at >= 2.58 else "**" if at >= 1.96 else "*" if at >= 1.65 else ""


def regression_table(models, annualize_const=False):
    rows, all_params = [], []
    for m in models.values():
        all_params += list(m.params.index)
    idx = []
    for p in all_params:
        if p not in idx:
            idx.append(p)
    for p in idx:
        row = {"Variable": p}
        for name, m in models.items():
            if p in m.params.index:
                coef = m.params[p]
                if annualize_const and p == "const":
                    coef *= 12
                t = m.tvalues[p]
                row[name] = f"{coef:.4f}{stars(t)}({t:.2f})"
            else:
                row[name] = ""
        rows.append(row)
    row = {"Variable": "Obs"}
    for name, m in models.items():
        row[name] = int(m.nobs)
    rows.append(row)
    row = {"Variable": "Adj. R2"}
    for name, m in models.items():
        row[name] = f"{m.rsquared_adj:.3f}"
    rows.append(row)
    return pd.DataFrame(rows)


def perf_stats(r, ann=12):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return {}
    cum = (1 + r).cumprod()
    dd = cum / cum.cummax() - 1
    return {
        "N": len(r),
        "Mean_ann": r.mean() * ann,
        "Vol_ann": r.std() * np.sqrt(ann),
        "Sharpe": (r.mean() * ann) / (r.std() * np.sqrt(ann)) if r.std() > 0 else np.nan,
        "Skew": r.skew(),
        "Kurtosis": r.kurtosis(),
        "MaxDrawdown": dd.min(),
        "WorstMonth": r.min(),
        "BestMonth": r.max(),
        "CumReturn": cum.iloc[-1] - 1,
    }


def ff5_alpha(r, factors, lags=6):
    df = pd.concat([r.rename("ret"), factors[["Mkt_RF", "SMB", "HML", "RMW", "CMA", "RF"]]], axis=1).dropna()
    # The zero-cost MOM spread is already an excess/long-short return: do NOT subtract RF again.
    y = df["ret"]
    X = df[["Mkt_RF", "SMB", "HML", "RMW", "CMA"]]
    m = nw_reg(y, X, lags=lags)
    return m.params.get("const", np.nan) * 12, m.tvalues.get("const", np.nan), m.rsquared_adj, len(df)



def annualized_sharpe(x, ann=12):
    x = pd.Series(x).dropna()
    return (x.mean() * ann) / (x.std() * np.sqrt(ann)) if len(x) > 1 and x.std() > 0 else np.nan


def moving_block_bootstrap_sharpe_diff(strategy, benchmark, block=6, reps=5000, seed=42, ann=12):
    """Paired circular moving-block bootstrap for annualized Sharpe-ratio difference.

    Vectorized implementation for speed. One-sided p-value tests H1: strategy Sharpe > benchmark Sharpe.
    """
    df = pd.concat([pd.Series(strategy).rename("strategy"), pd.Series(benchmark).rename("benchmark")], axis=1).dropna()
    n = len(df)
    if n <= block + 1:
        return np.nan, np.nan, np.nan, n
    rng = np.random.default_rng(seed)
    vals = df.to_numpy(dtype=float)
    obs = annualized_sharpe(vals[:, 0], ann) - annualized_sharpe(vals[:, 1], ann)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(reps, n_blocks))
    block_offsets = np.arange(block)
    idx = (starts[:, :, None] + block_offsets[None, None, :]) % n
    idx = idx.reshape(reps, n_blocks * block)[:, :n]
    sample = vals[idx, :]  # reps x n x 2
    mean = sample.mean(axis=1) * ann
    std = sample.std(axis=1, ddof=1) * np.sqrt(ann)
    sharpe = mean / std
    diffs = sharpe[:, 0] - sharpe[:, 1]
    p_one_sided = float(np.mean(diffs <= 0))
    ci_low, ci_high = np.quantile(diffs, [0.025, 0.975])
    return obs, p_one_sided, (ci_low, ci_high), n


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

## 2. Load raw data and inspect files

In [ ]:
paths = {
    "mom": find_input_file(["F-F_Momentum_Factor_monthly.csv", "F-F_Momentum_Factor_monthly*.csv"]),
    "ff5m": find_input_file(["F-F_Research_Data_5_Factors_monthly.csv", "F-F_Research_Data_5_Factors_monthl;y.csv", "F-F_Research_Data_5_Factors_month*.csv"]),
    "ff5d": find_input_file(["F-F_Research_Data_5_Factors_daily.csv", "F-F_Research_Data_5_Factors_daily*.csv"]),
    "vix": find_input_file(["VIXCLS.csv", "VIXCLS*.csv"]),
    "sent": find_input_file(["BW_SENTIMENT.xlsx", "BW_SENTIMENT*.xlsx"]),
    # Optional: if you download the Fama-French 3-factor monthly file starting in 1926, put it here.
    # The notebook will then fill pre-1963 MKT and Bear states in Table 3 instead of leaving them blank.
    "ff3m_optional": find_input_file(["F-F_Research_Data_Factors_monthly.csv", "F-F_Research_Data_Factors*.csv"], required=False),
}
paths

mom = read_ff_monthly(paths["mom"]).rename(columns={"MOM": "MOM"})
ff5m = read_ff_monthly(paths["ff5m"]).rename(columns={"Mkt-RF": "Mkt_RF"})
ff5d = read_ff_daily(paths["ff5d"]).rename(columns={"Mkt-RF": "Mkt_RF"})
ff3m = read_optional_ff3_monthly(paths["ff3m_optional"])

print("Momentum monthly:", mom.Date.min(), mom.Date.max(), mom.shape)
print("FF5 monthly:", ff5m.Date.min(), ff5m.Date.max(), ff5m.shape)
print("FF5 daily:", ff5d.Date.min(), ff5d.Date.max(), ff5d.shape)
if ff3m is not None:
    print("Optional FF3 monthly used for full-sample market states:", ff3m.Date.min(), ff3m.Date.max(), ff3m.shape)
else:
    print("Optional FF3 monthly not found. Pre-1963 market-state variables will remain missing, not zero.")

mom.head(), ff5m.head(), ff5d.head()

## 3. Construct realized variance, VIX, and sentiment variables

In [ ]:
# RV(t) = 12 × SUM_{d in month t} [daily total market return]^2.
# This is monthly realized variance annualized by 12, NOT a rolling 126-day ×252 estimator.
# Daily MARKET data underlie the aggregate monthly RV predictor.
ff5d["MKT_DAILY"] = ff5d["Mkt_RF"] + ff5d["RF"]
ff5d["Month"] = ff5d["Date"] + pd.offsets.MonthEnd(0)
rv = (
    ff5d.groupby("Month")
    .agg(
        RV=("MKT_DAILY", lambda x: np.sum(x.dropna() ** 2)),
        RV_ann=("MKT_DAILY", lambda x: np.sum(x.dropna() ** 2) * 12),
        RV_sqrt_ann=("MKT_DAILY", lambda x: np.sqrt(np.sum(x.dropna() ** 2) * 12)),
    )
    .reset_index()
    .rename(columns={"Month": "Date"})
)

# VIX: FRED starts in 1990, so all VIX-based tests are naturally post-1990.
vix = pd.read_csv(paths["vix"])
vix.columns = [c.strip() for c in vix.columns]
if "date" not in vix.columns:
    if "DATE" in vix.columns:
        vix = vix.rename(columns={"DATE": "date"})
    elif "observation_date" in vix.columns:
        vix = vix.rename(columns={"observation_date": "date"})
if "VIX" not in vix.columns:
    # Common FRED column name
    cand = [c for c in vix.columns if c.upper() in ["VIXCLS", "VIX"]]
    if cand:
        vix = vix.rename(columns={cand[0]: "VIX"})
vix["date"] = pd.to_datetime(vix["date"], errors="coerce")
vix["VIX"] = pd.to_numeric(vix["VIX"], errors="coerce")
vix = vix.dropna(subset=["date"])
vix["Month"] = vix["date"] + pd.offsets.MonthEnd(0)
vixm = (
    vix.groupby("Month")
    .agg(VIX=("VIX", "last"), VIX_avg=("VIX", "mean"), VIX_max=("VIX", "max"))
    .reset_index()
    .rename(columns={"Month": "Date"})
)

# Baker-Wurgler sentiment
sent = pd.read_excel(paths["sent"], sheet_name="DATA")
sent = sent[pd.to_numeric(sent["yearmo"], errors="coerce").notna()].copy()
sent["Date"] = sent["yearmo"].astype(int).apply(month_end_from_yyyymm)
sent = sent[["Date", "SENT", "SENT_ORTH", "recess"]].copy()
sent["SENT"] = pd.to_numeric(sent["SENT"], errors="coerce")
sent["SENT_ORTH"] = pd.to_numeric(sent["SENT_ORTH"], errors="coerce")

# Main sentiment signal: six-month moving average of orthogonalized sentiment
sent["SENT6"] = sent["SENT_ORTH"].rolling(6).mean()
sent["SENT_CHG6"] = sent["SENT6"] - sent["SENT6"].shift(6)


## 4. Merge master monthly dataset and construct predictors

In [ ]:
# Market return for FF5 controls
ff5m["MKT_FF5"] = ff5m["Mkt_RF"] + ff5m["RF"]

# Best available market return for state variables.
# If the optional FF3 monthly file is supplied, this extends MKT_STATE back to 1926.
if ff3m is not None:
    ff3_state = ff3m[["Date", "Mkt_RF", "RF"]].copy()
    ff3_state["MKT_STATE"] = ff3_state["Mkt_RF"] + ff3_state["RF"]
    state_mkt = ff3_state[["Date", "MKT_STATE"]]
else:
    state_mkt = ff5m[["Date", "MKT_FF5"]].rename(columns={"MKT_FF5": "MKT_STATE"})

master = (
    mom.merge(ff5m, on="Date", how="left")
    .merge(state_mkt, on="Date", how="left")
    .merge(rv, on="Date", how="left")
    .merge(vixm, on="Date", how="left")
    .merge(sent, on="Date", how="left")
)
master = master.sort_values("Date").set_index("Date")

# MKT is the best available total market return used for descriptive tables and state variables.
master["MKT"] = master["MKT_STATE"]

# One-month-ahead momentum return target
master["MOM_lead"] = master["MOM"].shift(-1)

# D&M bear indicator: lagged 24-month cumulative market return is negative.
# Important fix: when market data are unavailable, Bear remains NaN rather than being coded as 0.
master["MKT_24m_cum"] = (1 + master["MKT"]).rolling(24, min_periods=24).apply(np.prod, raw=True) - 1
master["Bear"] = np.where(master["MKT_24m_cum"].notna(), (master["MKT_24m_cum"] < 0).astype(float), np.nan)

# Rolling momentum variance for strategy weights
master["MOM_vol12"] = master["MOM"].rolling(12).std()
master["MOM_var12"] = master["MOM_vol12"] ** 2

# Expanding VIX 80th percentile: avoids using future VIX data in OOS and in-sample tests.
# The first 60 VIX observations are used only to initialize the threshold.
# Fast manual expanding 80th percentile using only previous VIX observations
vix_values = master["VIX"].to_numpy(dtype=float)
q80_exp = np.full(len(vix_values), np.nan)
prev_vix = []
for i, val in enumerate(vix_values):
    if len(prev_vix) >= 60:
        q80_exp[i] = np.nanquantile(prev_vix, 0.8)
    if not np.isnan(val):
        prev_vix.append(val)
master["VIX_q80_exp"] = q80_exp
master["VIX_high"] = np.where(master["VIX_q80_exp"].notna() & master["VIX"].notna(), (master["VIX"] > master["VIX_q80_exp"]).astype(float), np.nan)

# Composite is defined only when both Bear and VIX_high are observed.
# This fixes the old sample inconsistency where VIX-unavailable months could enter as false zeros.
master["Composite"] = np.nan
mask_comp = master["Bear"].notna() & master["VIX_high"].notna()
master.loc[mask_comp, "Composite"] = ((master.loc[mask_comp, "Bear"] == 1) | (master.loc[mask_comp, "VIX_high"] == 1)).astype(float)

# A descriptive full-threshold VIX dummy is kept only for diagnostics, not for regressions/trading.
q80 = master.loc[master["VIX"].notna(), "VIX"].quantile(0.8)
master["VIX_high_full"] = np.where(master["VIX"].notna(), (master["VIX"] > q80).astype(float), np.nan)
master["Composite_full_diag"] = np.nan
mask_comp_full = master["Bear"].notna() & master["VIX_high_full"].notna()
master.loc[mask_comp_full, "Composite_full_diag"] = ((master.loc[mask_comp_full, "Bear"] == 1) | (master.loc[mask_comp_full, "VIX_high_full"] == 1)).astype(float)

master["Bear_RV"] = master["Bear"] * master["RV_ann"]
master["Composite_RV"] = master["Composite"] * master["RV_ann"]

master.to_csv(os.path.join(CLEAN, "master_monthly.csv"))
master.tail()

## 5. Data coverage and summary statistics

In [ ]:
coverage = []
for col in ["MOM", "MKT", "Mkt_RF", "RF", "RV_ann", "VIX", "SENT_ORTH", "SENT6", "Bear", "Composite"]:
    s = master[col].dropna()
    coverage.append({
        "Variable": col,
        "Start": s.index.min().strftime("%Y-%m") if len(s) else "",
        "End": s.index.max().strftime("%Y-%m") if len(s) else "",
        "N": len(s),
    })
coverage = pd.DataFrame(coverage)
coverage.to_csv(os.path.join(TAB, "table_01_data_coverage.csv"), index=False)
coverage

In [ ]:
summary_cols = ["MOM", "MKT", "RV_sqrt_ann", "VIX", "SENT_ORTH", "SENT6"]
sumrows = []
for c in summary_cols:
    s = master[c].dropna()
    sumrows.append({
        "Variable": c,
        "N": len(s),
        "Mean": s.mean(),
        "Std": s.std(),
        "Min": s.min(),
        "P25": s.quantile(.25),
        "Median": s.median(),
        "P75": s.quantile(.75),
        "Max": s.max(),
    })
summary = pd.DataFrame(sumrows)
summary.to_csv(os.path.join(TAB, "table_02_summary_statistics.csv"), index=False)
summary

## 6. Worst momentum months and crash-state indicators

In [ ]:
worst = master.dropna(subset=["MOM"]).sort_values("MOM").head(20)
worst_table = worst[["MOM", "MKT", "MKT_24m_cum", "Bear", "VIX", "VIX_high", "Composite", "SENT6"]].copy()
# False-zero fix: unavailable variables remain blank/NaN in the CSV and notebook output.
worst_table.reset_index().to_csv(os.path.join(TAB, "table_03_worst_momentum_months.csv"), index=False)
worst_table

## 7. In-sample predictive regressions
**Date convention:** predictive regressions are indexed by signal-formation month t (dependent variable MOM at t+1); the post-2013 regression window Jan-2014–Dec-2023 is followed by returns Feb-2014–Jan-2024. OOS portfolio tables below are instead indexed by realized return month.


In [ ]:
# Consistent VIX-era in-sample sample.
# All model columns are estimated on the same observations and use the same real-time expanding Composite definition.
regdata = master.dropna(subset=["MOM_lead", "Bear", "RV_ann", "Composite", "SENT6"]).copy()

models = {
    "Hist mean": nw_reg(regdata["MOM_lead"], pd.DataFrame(index=regdata.index), 6),
    "DM": nw_reg(regdata["MOM_lead"], regdata[["Bear", "RV_ann", "Bear_RV"]], 6),
    "Composite": nw_reg(regdata["MOM_lead"], regdata[["Composite", "RV_ann"]], 6),
    "Sentiment": nw_reg(regdata["MOM_lead"], regdata[["SENT6"]], 6),
    "Combined": nw_reg(regdata["MOM_lead"], regdata[["Composite", "RV_ann", "SENT6"]], 6),
}
regtab = regression_table(models)
regtab.to_csv(os.path.join(TAB, "table_04_predictive_regressions_insample.csv"), index=False)
print("Table 4 common in-sample period:", regdata.index.min().strftime("%Y-%m"), "to", regdata.index.max().strftime("%Y-%m"), "N=", len(regdata))
regtab

In [ ]:
reg_post = regdata[regdata.index >= pd.Timestamp("2014-01-31")]
models_post = {
    "DM": nw_reg(reg_post["MOM_lead"], reg_post[["Bear", "RV_ann", "Bear_RV"]], 3),
    "Composite": nw_reg(reg_post["MOM_lead"], reg_post[["Composite", "RV_ann"]], 3),
    "Sentiment": nw_reg(reg_post["MOM_lead"], reg_post[["SENT6"]], 3),
    "Combined": nw_reg(reg_post["MOM_lead"], reg_post[["Composite", "RV_ann", "SENT6"]], 3),
}
regtab_post = regression_table(models_post)
regtab_post.to_csv(os.path.join(TAB, "table_05_predictive_regressions_post2013.csv"), index=False)
print("Table 5 post-2013 period:", reg_post.index.min().strftime("%Y-%m"), "to", reg_post.index.max().strftime("%Y-%m"), "N=", len(reg_post))
regtab_post

## 8. Expanding-window out-of-sample forecast horse race

In [ ]:
# OOS forecasting: predictor information through month t forecasts the RETURN in t+1.
# The benchmark and ALL models use exactly the same training rows for the common comparison.
# VIX high is constructed from PRIOR VIX thresholds; monthly VIX at end-t is available for t+1.
oos_base = master.dropna(subset=["MOM_lead", "Bear", "RV_ann", "Bear_RV", "Composite", "SENT6", "MOM_var12"]).copy()
oos_base = oos_base[oos_base.index >= pd.Timestamp("1995-01-31")]
models_spec = {
    "Historical mean": [],
    "DM": ["Bear", "RV_ann", "Bear_RV"],
    "Composite": ["Composite", "RV_ann"],
    "Sentiment": ["SENT6"],
    "Combined": ["Composite", "RV_ann", "SENT6"],
}
# The first forecast for Jan-2005 forms in Dec-2004: 119 target-observed rows
# (Jan-1995 to Nov-2004) are available. The Dec-2004 target is only seen in Jan-2005.
min_train = 119
pred = pd.DataFrame(index=oos_base.index, columns=list(models_spec.keys()), dtype=float)
actual = oos_base["MOM_lead"].copy()
for i in range(min_train, len(oos_base)):
    train = oos_base.iloc[:i]
    test = oos_base.iloc[i]
    for name, xs in models_spec.items():
        if len(xs) == 0:
            pred.loc[oos_base.index[i], name] = train["MOM_lead"].mean()
        else:
            m = sm.OLS(train["MOM_lead"], sm.add_constant(train[xs], has_constant="add")).fit()
            xx = pd.DataFrame([test[xs].values], columns=xs, index=[oos_base.index[i]])
            xx = sm.add_constant(xx, has_constant="add")
            pred.loc[oos_base.index[i], name] = float(m.predict(xx).iloc[0])
forecast_all = pd.concat([actual.rename("Actual"), pred], axis=1).dropna()
forecast_all["Formation_month"] = forecast_all.index
forecast_all.index = forecast_all.index + pd.offsets.MonthEnd(1)
forecast_all.index.name = "Realized_month"
oos = forecast_all.loc["2005-01-31":"2023-12-31"].drop(columns="Formation_month").copy()
oos.to_csv(os.path.join(CLEAN, "oos_forecasts.csv"))
print("Common OOS realized-return dates:", oos.index.min().date(), "through", oos.index.max().date(), "N=",len(oos))
print("Training first month: 1995-01 (signal); last observed target before first forecast: 2004-12")
oos.head(),oos.tail()


In [ ]:
base_err = oos["Actual"] - oos["Historical mean"]
metrics = []
for name in models_spec.keys():
    err = oos["Actual"] - oos[name]
    r2 = 1 - (err.pow(2).sum() / base_err.pow(2).sum()) if name != "Historical mean" else 0.0
    mae = err.abs().mean()
    rmse = np.sqrt(err.pow(2).mean())
    corr = oos[["Actual", name]].corr().iloc[0, 1]
    if name != "Historical mean":
        f = base_err.pow(2) - (err.pow(2) - (oos[name] - oos["Historical mean"]).pow(2))
        t = float(sm.OLS(f, np.ones((len(f),1))).fit(cov_type="HAC",cov_kwds={"maxlags":6}).tvalues.iloc[0]) if len(f)>6 else np.nan
        p = 1 - stats.norm.cdf(t) if pd.notna(t) else np.nan
    else:
        t, p = np.nan, np.nan
    metrics.append({
        "Model": name,
        "OOS_Start": oos.index.min().strftime("%Y-%m"),
        "OOS_End": oos.index.max().strftime("%Y-%m"),
        "N": len(oos),
        "OOS_R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "Forecast_Corr": corr,
        "Clark_West_t": t,
        "Clark_West_p_one_sided": p,
    })
metricstab = pd.DataFrame(metrics)
metricstab.to_csv(os.path.join(TAB, "table_06_oos_forecast_metrics.csv"), index=False)
metricstab

## 9. Crash classification (retrospective 5% tail reference)

This retained top-5% event definition is **ex post** and diagnostic, not a claim that a deployable warning system knew the sample quantile. The additional pre-2005 fixed threshold and chance-warning controls below constitute the investable-cutoff sensitivity analysis.


In [ ]:
crash_q = oos["Actual"].quantile(0.05)
crash = (oos["Actual"] <= crash_q).astype(int)
class_rows = []
for name in models_spec.keys():
    # A simple crash warning is a negative forecasted momentum premium.
    sig = (oos[name] < 0).astype(int)
    TP = int(((sig == 1) & (crash == 1)).sum())
    FP = int(((sig == 1) & (crash == 0)).sum())
    FN = int(((sig == 0) & (crash == 1)).sum())
    TN = int(((sig == 0) & (crash == 0)).sum())
    precision = TP / (TP + FP) if TP + FP > 0 else np.nan
    recall = TP / (TP + FN) if TP + FN > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0 else np.nan
    class_rows.append({
        "Model": name,
        "Crash_threshold_5pct": crash_q,
        "Warnings": int(sig.sum()),
        "Crash_months": int(crash.sum()),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })
classtab = pd.DataFrame(class_rows)
classtab.to_csv(os.path.join(TAB, "table_07_crash_classification.csv"), index=False)
classtab

## 10. OOS dynamic strategies and FF5 alphas

In [ ]:
strat = pd.DataFrame(index=oos.index)
strat["Static MOM"] = oos["Actual"]
# Signal at end (t-1) determines exposure to return in month t.
formation_dates = oos.index - pd.offsets.MonthEnd(1)
vol = oos_base.loc[formation_dates, "MOM_vol12"].copy()
vol.index = oos.index
var = oos_base.loc[formation_dates, "MOM_var12"].replace(0, np.nan).copy()
var.index = oos.index
# Ex ante target defined using the observation-available initial estimation window.
target_vol = oos_base.iloc[:min_train]["MOM_lead"].std()
bsc_w = (target_vol / vol).clip(0, 2)
strat["BSC vol-managed"] = bsc_w * oos["Actual"]
weight_data = pd.DataFrame(index=oos.index)
weight_data["BSC vol-managed"] = bsc_w
for name in ["DM", "Composite", "Sentiment", "Combined"]:
    w = (oos[name] / var).replace([np.inf, -np.inf], np.nan).clip(-1, 2)
    strat[name + " dynamic"] = w * oos["Actual"]
    weight_data[name + " dynamic"] = w
    wlong = (oos[name] / var).replace([np.inf, -np.inf], np.nan).clip(0, 2)
    strat[name + " long-only dynamic"] = wlong * oos["Actual"]
    weight_data[name + " long-only dynamic"] = wlong
strat.to_csv(os.path.join(CLEAN, "strategy_returns_oos.csv"))
weight_data.to_csv(os.path.join(CLEAN, "strategy_weights_oos.csv"))
strat.head()


In [ ]:
perf = []
for c in strat.columns:
    d = perf_stats(strat[c])
    d["Strategy"] = c
    alpha, t, r2, n = ff5_alpha(strat[c], master, lags=6)
    d["FF5_alpha_ann"] = alpha
    d["t_alpha"] = t
    d["FF5_adj_R2"] = r2
    perf.append(d)
perftab = pd.DataFrame(perf).set_index("Strategy")
order = ["N", "Mean_ann", "Vol_ann", "Sharpe", "Skew", "MaxDrawdown", "WorstMonth", "BestMonth", "CumReturn", "FF5_alpha_ann", "t_alpha", "FF5_adj_R2"]
perftab = perftab[order]
perftab.to_csv(os.path.join(TAB, "table_08_strategy_performance_oos.csv"))
perftab
print("OOS perf done")

In [ ]:
perfp = []
strat_post = strat[strat.index >= pd.Timestamp("2014-01-31")]
for c in strat_post.columns:
    d = perf_stats(strat_post[c])
    d["Strategy"] = c
    alpha, t, r2, n = ff5_alpha(strat_post[c], master, lags=3)
    d["FF5_alpha_ann"] = alpha
    d["t_alpha"] = t
    d["FF5_adj_R2"] = r2
    perfp.append(d)
perftab_post = pd.DataFrame(perfp).set_index("Strategy")[order]
perftab_post.to_csv(os.path.join(TAB, "table_09_strategy_performance_post2013.csv"))
perftab_post
print("Post2013 perf done")

## 10.1 Paired Sharpe-ratio uncertainty: descriptive and multiplicity-aware

One-sided bootstrap tail probabilities and percentile intervals answer different inferential questions; do not declare an improvement significant when a corresponding two-sided interval spans zero. With nine comparisons, Holm-adjusted one-sided p-values are also reported below.


In [ ]:
# Formal Sharpe-ratio difference tests for the OOS strategy table.
# This addresses the referee concern that Sharpe-ratio claims must be tested rather than only described.
benchmarks = ["Static MOM", "BSC vol-managed"]
test_strategies = [
    "BSC vol-managed",
    "DM long-only dynamic",
    "Composite long-only dynamic",
    "Sentiment long-only dynamic",
    "Combined long-only dynamic",
]
sharpe_tests = []
for bench in benchmarks:
    for strat_name in test_strategies:
        if strat_name == bench:
            continue
        diff, p, ci, n = moving_block_bootstrap_sharpe_diff(strat[strat_name], strat[bench], block=6, reps=5000, seed=42)
        sharpe_tests.append({
            "Strategy": strat_name,
            "Benchmark": bench,
            "N": n,
            "Sharpe_strategy": annualized_sharpe(strat[strat_name]),
            "Sharpe_benchmark": annualized_sharpe(strat[bench]),
            "Sharpe_diff": diff,
            "p_one_sided_strategy_gt_benchmark": p,
            "CI_2.5%": ci[0] if isinstance(ci, tuple) else np.nan,
            "CI_97.5%": ci[1] if isinstance(ci, tuple) else np.nan,
            "Block_length_months": 6,
            "Bootstrap_reps": 5000,
        })
sharpe_test_tab = pd.DataFrame(sharpe_tests)
sharpe_test_tab.to_csv(os.path.join(TAB, "table_10_sharpe_difference_tests.csv"), index=False)
sharpe_test_tab
print("Sharpe tests done")

## 11. Figures

In [ ]:
# Figure 1: full-sample cumulative momentum
cum = (1 + master["MOM"].dropna()).cumprod()
plt.figure(figsize=(10, 5))
plt.plot(cum.index, cum.values)
plt.yscale("log")
plt.title("Cumulative performance of Fama-French momentum factor")
plt.ylabel("Growth of $1 (log scale)")
plt.xlabel("Date")
savefig(os.path.join(FIG, "fig_01_cumulative_momentum_full_sample.png"))

# Figure 2: OOS drawdown
cum_oos = (1 + strat["Static MOM"].dropna()).cumprod()
dd = cum_oos / cum_oos.cummax() - 1
plt.figure(figsize=(10, 4))
plt.plot(dd.index, dd.values)
plt.title("Momentum drawdowns during OOS period")
plt.ylabel("Drawdown")
plt.xlabel("Date")
savefig(os.path.join(FIG, "fig_02_oos_momentum_drawdown.png"))

# Figure 3: VIX and sentiment
p = master[["VIX", "SENT6"]].dropna(subset=["VIX"])
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(p.index, p["VIX"], label="VIX")
ax1.set_ylabel("VIX")
ax2 = ax1.twinx()
ax2.plot(p.index, p["SENT6"], label="Sentiment 6M", linestyle="--")
ax2.set_ylabel("Sentiment 6M")
plt.title("Fast volatility and sentiment predictors")
plt.legend(fontsize=8)
savefig(os.path.join(FIG, "fig_03_predictors_vix_sentiment.png"))

# Figure 4: cumulative strategy returns
sel = ["Static MOM", "BSC vol-managed", "DM long-only dynamic", "Composite long-only dynamic", "Sentiment long-only dynamic", "Combined long-only dynamic"]
plt.figure(figsize=(10, 6))
for c in sel:
    s = (1 + strat[c].dropna()).cumprod()
    plt.plot(s.index, s.values, label=c)
plt.yscale("log")
plt.legend(fontsize=8)
plt.title("OOS cumulative returns of momentum timing strategies")
plt.ylabel("Growth of $1 (log scale)")
plt.xlabel("Date")
savefig(os.path.join(FIG, "fig_04_oos_strategy_cumulative_returns.png"))

# Figure 5: OOS Sharpe ratios
sh = perftab.loc[sel, "Sharpe"].dropna().sort_values()
plt.figure(figsize=(9, 4))
plt.barh(sh.index, sh.values)
plt.title("OOS Sharpe ratios")
plt.xlabel("Annualized Sharpe ratio")
savefig(os.path.join(FIG, "fig_05_oos_sharpe_ratios.png"))

# Figure 6: worst momentum months
w15 = worst_table.head(15).copy()
labels = [d.strftime("%Y-%m") for d in w15.index]
plt.figure(figsize=(10, 5))
plt.bar(labels, w15["MOM"].values)
plt.xticks(rotation=60, ha="right")
plt.title("Worst Fama-French momentum months")
plt.ylabel("Monthly return")
savefig(os.path.join(FIG, "fig_06_worst_momentum_months.png"))

print("Saved figures to", FIG)

## 12. Matched-sample base rates, ranking discrimination, and fixed historical crash threshold

We retain Table 7's OOS 5% tail classification only as a descriptive benchmark. The robust alternative fixes the 5th percentile from **pre-2005 momentum returns in the same market/VIX-era estimation window** and evaluates 2005–2023 without re-selecting the cutoff. For a randomly selected warning month, expected precision equals the contemporaneous crash base rate. ROC AUC measures ranking rather than an arbitrary warning cutoff; 0.5 is random discrimination. Hypergeometric right-tail p-values condition on the observed number of warnings and crash events and are descriptive, not adjusted for temporal dependence or model searching.

In [ ]:
from scipy.stats import hypergeom, rankdata

def auc_score(y, score):
    y = np.asarray(y,dtype=int)
    score = np.asarray(score,dtype=float)
    positive = y.sum()
    negative = len(y)-positive
    if positive==0 or negative==0:
        return np.nan
    ranks = rankdata(score, method="average")
    return (ranks[y==1].sum()-positive*(positive+1)/2)/(positive*negative)

# Confirm no look-ahead in crash cutoff: only realized momentum through Dec-2004.
train_return_start = pd.Timestamp("1995-02-28")
prior_momentum = master.loc[train_return_start:"2004-12-31", "MOM"].dropna()
cutoff_pre2005 = float(prior_momentum.quantile(0.05))
thresholds = {"ex_post_5pct_reference":float(oos["Actual"].quantile(.05)),
              "fixed_1995_2004_5pct":cutoff_pre2005}
rows=[]
for label,cutoff in thresholds.items():
    event=(oos["Actual"]<=cutoff).astype(int)
    prevalence=float(event.mean())
    for name in models_spec:
        alarm=(oos[name]<0).astype(int)
        tp=int((alarm*event).sum())
        k=int(alarm.sum())
        m=int(event.sum())
        rows.append({"Event_definition":label,"Model":name,"Threshold":cutoff,"OOS_months":len(event),
                     "Crashes":m,"Base_rate":prevalence,"Warnings":k,"TP":tp,
                     "Precision":tp/k if k else np.nan,"Precision_lift":(tp/k)/prevalence if k and prevalence else np.nan,
                     "Recall":tp/m if m else np.nan,"AUC":auc_score(event,-oos[name]),
                     "Random_alarm_p":hypergeom.sf(tp-1,len(event),m,k) if k else np.nan})
auc_tab=pd.DataFrame(rows)
auc_tab.to_csv(os.path.join(TAB,"table_11_crash_auc_base_rate.csv"),index=False)
print("Pre-OOS fixed cutoff",round(cutoff_pre2005,4),"training N",len(prior_momentum))
display(auc_tab.round(3))


## Exposure constraints and alternative rolling monthly MOM windows (2005–2023 companion analysis)

The monthly-volatility rule follows σ-target divided by a prior 12-month standard deviation of monthly MOM factor returns. This specifies an aggregate factor-level exposure rule, not the original security-level construction. The target is the sample standard deviation of the 119 realized MOM returns from February 1995 through December 2004 (the training sample); we report its value explicitly. Inference about financing and implementation requires additional assumptions.

In [ ]:
diagnostics=[]
for name in weight_data.columns:
    w=weight_data[name].dropna()
    r=strat[name].dropna()
    diagnostics.append({"Strategy":name,"N":len(w),"Mean_weight":w.mean(),"Mean_abs_weight":w.abs().mean(),
                        "Share_at_plus2":(w>=2-1e-10).mean(),"Share_at_zero":(w.abs()<=1e-10).mean(),
                        "Share_negative":(w<0).mean(),"Vol_ann":r.std()*np.sqrt(12),
                        "Max_drawdown":perf_stats(r)["MaxDrawdown"],"Sharpe":annualized_sharpe(r)})
leverage_tab=pd.DataFrame(diagnostics)
leverage_tab.to_csv(os.path.join(TAB,"table_12_exposure_diagnostics.csv"),index=False)
# Alternative cap/estimation sensitivity (same ex ante pre-OOS target; no retuning).
variants={"Static MOM":strat["Static MOM"],"BSC monthly12 cap2":strat["BSC vol-managed"]}
variants["BSC monthly12 cap1"]=((target_vol/vol).clip(0,1)*oos["Actual"])
for wlen in [6,24]:
    altvol=master["MOM"].rolling(wlen).std().loc[formation_dates].copy()
    altvol.index=oos.index
    variants[f"BSC monthly{wlen} cap1"]=(target_vol/altvol).clip(0,1)*oos["Actual"]
    variants[f"BSC monthly{wlen} cap2"]=(target_vol/altvol).clip(0,2)*oos["Actual"]
variants["DM long-only cap1"]=(oos["DM"]/var).clip(0,1)*oos["Actual"]
variants["Combined long-only cap1"]=(oos["Combined"]/var).clip(0,1)*oos["Actual"]
sens=[]
for name,r in variants.items():
    p=perf_stats(r)
    sens.append({"Strategy":name,"N":p["N"],"Mean_ann":p["Mean_ann"],"Vol_ann":p["Vol_ann"],
                 "Sharpe":p["Sharpe"],"MaxDrawdown":p["MaxDrawdown"],"CumReturn":p["CumReturn"]})
sensitivity_tab=pd.DataFrame(sens)
sensitivity_tab.to_csv(os.path.join(TAB,"table_13_vol_scaling_sensitivity.csv"),index=False)
display(leverage_tab.round(3),sensitivity_tab.round(3))


## 14. Nine-test Sharpe multiplicity, Sharpe two-sided intervals, and factor alpha

Holm corrections use the original family of nine one-sided Sharpe-difference bootstrap tests; these corrections are within-family conservative. A two-sided percentile confidence interval including zero indicates lack of standard two-sided support. Factor alpha is correctly calculated on the zero-cost momentum premium without deducting the risk-free rate a second time.

In [ ]:
from statsmodels.stats.multitest import multipletests
sharpe_test_tab["Holm_p_one_sided"] = multipletests(sharpe_test_tab["p_one_sided_strategy_gt_benchmark"], method="holm")[1]
sharpe_test_tab["Two_sided_CI_excludes_zero"]=(sharpe_test_tab["CI_2.5%"]>0)|(sharpe_test_tab["CI_97.5%"]<0)
sharpe_test_tab.to_csv(os.path.join(TAB,"table_10_sharpe_difference_tests.csv"),index=False)
display(sharpe_test_tab.round(4))


## 2005–2026 primary sample construction

The 256-month main sample covers all market-based signals through April 2026. The VIX-based common-period initialization is January 1995; a second DM model expands the available training history back to January 1966 and uses its *own sample-matched historical-mean benchmark* in the forecast comparison. The original 2005–2023 sentiment calculations above are supplementary only.

In [ ]:
market_only=master.dropna(subset=["MOM_lead","Bear","RV_ann","Bear_RV","Composite","MOM_var12"]).copy()
market_only=market_only.loc["1995-01-31":]
extspec={"Historical mean":[],"DM":["Bear","RV_ann","Bear_RV"],"Composite":["Composite","RV_ann"]}
ext_rows=[]
for i in range(119,len(market_only)):
    t=market_only.index[i]
    realized=t+pd.offsets.MonthEnd(1)
    if realized<pd.Timestamp("2005-01-31") or realized>pd.Timestamp("2026-04-30"):
        continue
    history=market_only.iloc[:i]
    line={"Date":realized,"Actual":market_only.iloc[i]["MOM_lead"]}
    for name,xs in extspec.items():
        if not xs: line[name]=history["MOM_lead"].mean()
        else:
            model=sm.OLS(history["MOM_lead"],sm.add_constant(history[xs],has_constant="add")).fit()
            now=sm.add_constant(pd.DataFrame([market_only.iloc[i][xs].values],columns=xs),has_constant="add")
            line[name]=float(model.predict(now).iloc[0])
    ext_rows.append(line)
extoos=pd.DataFrame(ext_rows).set_index("Date")
extstat=[]
for stop in ["2023-12-31","2026-04-30"]:
    dd=extoos.loc[:stop]
    err0=dd["Actual"]-dd["Historical mean"]
    for name in extspec:
        err=dd["Actual"]-dd[name]
        f=err0**2-err**2+(dd[name]-dd["Historical mean"])**2
        cw=float(sm.OLS(f,np.ones((len(f),1))).fit(cov_type="HAC",cov_kwds={"maxlags":6}).tvalues.iloc[0]) if name!="Historical mean" else np.nan
        extstat.append({"Evaluation_end":stop[:7],"Model":name,"N":len(dd),
                        "OOS_R2":1-(err**2).sum()/(err0**2).sum(),"RMSE":np.sqrt((err**2).mean()),
                        "CW_HAC_t":cw,"CW_p_one_sided":stats.norm.sf(cw) if name!="Historical mean" else np.nan})
extension_tab=pd.DataFrame(extstat)
extension_tab.to_csv(os.path.join(TAB,"table_14_market_only_extended_to_2026.csv"),index=False)
extoos.to_csv(os.path.join(CLEAN,"market_only_forecasts_to_2026.csv"))
# market-only vol-managed strategy extension using entirely observed data
formation_ext=extoos.index-pd.offsets.MonthEnd(1)
vol_ext=master.loc[formation_ext,"MOM_vol12"].copy()
vol_ext.index=extoos.index
w_ext=(target_vol/vol_ext).clip(0,2)
ret_ext=pd.DataFrame({"Static MOM":extoos["Actual"],"BSC monthly12 cap2":w_ext*extoos["Actual"]})
ext_performance=[]
for start in ["2005-01-31","2024-01-31"]:
    for name in ret_ext:
        r=ret_ext.loc[start:,name]
        pp=perf_stats(r)
        ext_performance.append({"Start":start[:7],"End":ret_ext.index[-1].strftime("%Y-%m"),"Strategy":name,
                                "N":pp["N"],"Mean_ann":pp["Mean_ann"],"Vol_ann":pp["Vol_ann"],
                                "Sharpe":pp["Sharpe"],"MaxDrawdown":pp["MaxDrawdown"]})
ext_performance_tab=pd.DataFrame(ext_performance)
ext_performance_tab.to_csv(os.path.join(TAB,"table_15_extended_simple_portfolio.csv"),index=False)
display(extension_tab.round(4),ext_performance_tab.round(4))


## 12. Key results

In [ ]:
print("Main common OOS sample:", oos.index.min().strftime("%Y-%m"), "to", oos.index.max().strftime("%Y-%m"), "N=", len(oos))
print("OOS forecast metrics:")
display(metricstab)
print("Selected strategy performance:")
display(perftab.loc[["Static MOM", "BSC vol-managed", "DM long-only dynamic", "Composite long-only dynamic", "Sentiment long-only dynamic", "Combined long-only dynamic"],
                    ["Mean_ann", "Vol_ann", "Sharpe", "MaxDrawdown", "FF5_alpha_ann", "t_alpha"]])
print("Sharpe-ratio difference tests:")
display(sharpe_test_tab)
print("All outputs saved in:", OUT)

## 16. Primary market-observable recursive forecasts, January 2005–April 2026

Each return for month t+1 is forecast using the end-of-t signals. Both variants of the historical mean are estimated on the **identical training targets** as their relevant models. The first usable DM formation observation is January 1966, before VIX is available. The fixed 1995-era comparison is reported alongside the longer DM history, preventing shorter composite availability from mechanically restricting the DM benchmark. The target volatility is independently specified by February 1995–December 2004 MOM returns, as in the accompanying historical 2005–2023 exercise.

In [ ]:
# Model-specific histories; each historical-mean comparator uses EXACTLY matching eligible targets.
main_specs = {
    'DM (1966 train)': (['Bear','RV_ann','Bear_RV'], '1966-01-31'),
    'DM (1995 train)': (['Bear','RV_ann','Bear_RV'], '1995-01-31'),
    'Composite (1995 train)': (['Composite','RV_ann'], '1995-01-31'),
}
main_oos = pd.DataFrame(index=pd.date_range('2005-01-31','2026-04-30',freq='ME'))
main_oos.index.name='Realized_month'
main_oos['Actual'] = master['MOM'].reindex(main_oos.index)
model_trainsets={}
for model_name,(xs,begin) in main_specs.items():
    training_rows=master.loc[begin:].dropna(subset=['MOM_lead']+xs).copy()
    model_trainsets[model_name]=training_rows
    for realized in main_oos.index:
        formation = realized-pd.offsets.MonthEnd(1)
        train=training_rows.loc[training_rows.index<formation]
        if len(train)<60 or formation not in training_rows.index:
            continue
        main_oos.loc[realized,'Mean matched: '+model_name] = train['MOM_lead'].mean()
        fitted=sm.OLS(train['MOM_lead'],sm.add_constant(train[xs],has_constant='add')).fit()
        this=sm.add_constant(training_rows.loc[[formation],xs],has_constant='add')
        main_oos.loc[realized,model_name] = float(fitted.predict(this).iloc[0])
    print(model_name, 'earliest eligible formation', training_rows.index[0].date(),
          'training N before 2005-01',len(training_rows.loc[:'2004-11-30']),
          'forecast nonmissing',main_oos[model_name].notna().sum())
assert main_oos.dropna().shape[0]==256
assert (main_oos['Actual'] == master.loc[main_oos.index,'MOM']).all()
main_oos.to_csv(os.path.join(CLEAN,'main_forecasts_2005_2026.csv'))
main_metrics=[]
for model_name in main_specs:
    bm='Mean matched: '+model_name
    sub=main_oos[['Actual',model_name,bm]].dropna()
    err0=sub['Actual']-sub[bm]; err=sub['Actual']-sub[model_name]
    cw=err0**2-err**2+(sub[model_name]-sub[bm])**2
    hac=sm.OLS(cw,np.ones((len(cw),1))).fit(cov_type='HAC',cov_kwds={'maxlags':6})
    main_metrics.append({'Model':model_name,'Training_start':main_specs[model_name][1][:7],
                         'N':len(sub),'OOS_R2':1-(err**2).sum()/(err0**2).sum(),
                         'RMSE':np.sqrt((err**2).mean()),'Benchmark_RMSE':np.sqrt((err0**2).mean()),
                         'MAE':err.abs().mean(),'CW_HAC_t':float(hac.tvalues.iloc[0]),
                         'CW_one_sided_p':stats.norm.sf(float(hac.tvalues.iloc[0]))})
main_metricstab=pd.DataFrame(main_metrics)
main_metricstab.to_csv(os.path.join(TAB,'MAIN_table_06_forecasts_2005_2026.csv'),index=False)
display(main_metricstab.round(4))


## 17. Crash prediction: fixed pre-evaluation threshold, AUC confidence intervals and quantile loss

The fixed event threshold is the fifth percentile of **February 1995–December 2004** MOM returns, set once before evaluation. A retrospective 5% tail within 2005–2026 is shown as a secondary discrimination assessment. We calculate circular six-month paired block-bootstrap percentile CIs for ROC AUC (3,000 repetitions, seeded); episodes are sparse, so intervals can be broad. The quantile-loss exercise builds conditional 5th-percentile forecasts from each recursively fitted regression mean plus the empirical fifth percentile of training residuals. Each model is compared with its same-training expanding historical-momentum unconditional 5% quantile. These are in-sample-calibrated distributions evaluated on genuinely subsequent returns; tail pinball loss is not a calibrated expected-shortfall test.

In [ ]:
from scipy.stats import hypergeom, rankdata

def auc_fast(event, scores):
    y=np.asarray(event,dtype=int)
    s=np.asarray(scores,dtype=float)
    n1=y.sum(); n0=len(y)-n1
    if n1==0 or n0==0:return np.nan
    return (rankdata(s)[y==1].sum()-n1*(n1+1)/2)/(n1*n0)

def auc_block_ci(y,score,reps=3000,block=6,seed=2718):
    rng=np.random.default_rng(seed)
    y=np.asarray(y);score=np.asarray(score);length=len(y)
    estimates=[]
    nblocks=int(np.ceil(length/block))
    for b in range(reps):
        starts=rng.integers(0,length,size=nblocks)
        idx=((starts[:,None]+np.arange(block))%length).flatten()[:length]
        this=auc_fast(y[idx],score[idx])
        if np.isfinite(this):estimates.append(this)
    return np.quantile(estimates,[.025,.975]),len(estimates)

fixed_q = float(master.loc['1995-02-28':'2004-12-31','MOM'].dropna().quantile(.05))
main_events = {'Fixed pre-2005 5%':fixed_q,
               'Retrospective OOS 5%':float(main_oos['Actual'].quantile(.05))}
auc_rows=[]
for event_name, threshold in main_events.items():
    y=(main_oos['Actual']<=threshold).astype(int)
    for model in main_specs:
        warn=(main_oos[model]<0).astype(int)
        n_alert=int(warn.sum()); tp=int((warn*y).sum()); prevalence=y.mean()
        auc=auc_fast(y,-main_oos[model]);ci,nn=auc_block_ci(y,-main_oos[model],seed=2718)
        auc_rows.append({'Definition':event_name,'Model':model,'Cutoff':threshold,'Crashes':int(y.sum()),
                         'Base_rate':prevalence,'Warnings':n_alert,'Hits':tp,'Precision':tp/n_alert if n_alert else np.nan,
                         'Precision_lift':(tp/n_alert/prevalence) if n_alert and prevalence else np.nan,
                         'Chance_p':hypergeom.sf(tp-1,len(y),int(y.sum()),n_alert) if n_alert else np.nan,
                         'AUC':auc,'AUC_CI_low':ci[0],'AUC_CI_high':ci[1], 'Valid_bootstraps':nn})
main_auctab=pd.DataFrame(auc_rows)
main_auctab.to_csv(os.path.join(TAB,'MAIN_table_07_crash_auc.csv'),index=False)
print('Fixed pre-2005 threshold',round(fixed_q,5));display(main_auctab.round(3))
# Conditional and unconditional tail quantile estimated in the same recursive expansion.
tail_rows=[]
for name,(xs,begin) in main_specs.items():
    dataset=model_trainsets[name]
    for realized in main_oos.index:
        formation=realized-pd.offsets.MonthEnd(1)
        hist=dataset.loc[dataset.index<formation]
        if len(hist)<60 or formation not in dataset.index:continue
        model=sm.OLS(hist['MOM_lead'],sm.add_constant(hist[xs],has_constant='add')).fit()
        cond=float(main_oos.loc[realized,name]+np.quantile(model.resid,.05))
        unconditional=float(hist['MOM_lead'].quantile(.05))
        y=float(main_oos.loc[realized,'Actual'])
        check=lambda q: (.05-float(y<q))*(y-q)
        tail_rows.append({'Date':realized,'Model':name,'Actual':y,'Cond_q05':cond,
                          'Uncond_q05':unconditional,'Loss_model':check(cond),'Loss_base':check(unconditional)})
tail_df=pd.DataFrame(tail_rows)
tail_df.to_csv(os.path.join(CLEAN,'tail_quantile_forecasts.csv'),index=False)
tail_stats=tail_df.groupby('Model').agg(N=('Actual','size'),Pinball_model=('Loss_model','mean'),Pinball_base=('Loss_base','mean')).reset_index()
tail_stats['Pinball_skill']=1-tail_stats['Pinball_model']/tail_stats['Pinball_base']
tail_stats.to_csv(os.path.join(TAB,'MAIN_tail_quantile_scoring.csv'),index=False)
display(tail_stats.round(5))


## 18. Primary strategy and constant-leverage placebo

All weights use information at t, returns at t+1. The target volatility is a **monthly standard deviation of 119 MOM returns (February 1995–December 2004)**, not 1927–2004. Volatility scaling uses the rolling *12 observed monthly returns* through t. The ex-post volatility-matched comparison rescales entire return histories by a single constant estimated on the evaluation sample; it is a diagnostic normalization, not an implementable trading strategy. Constant 1.6× MOM is a leverage-only placebo, so its gross Sharpe ratio must equal static MOM. No more than 1× exposure is used by cap1 strategies.

In [ ]:
# Primary strategy horse race: 256 dates, monthly formation convention.
formation_main=main_oos.index-pd.offsets.MonthEnd(1)
vol_main=master.loc[formation_main,'MOM_vol12'].copy();vol_main.index=main_oos.index
var_main=master.loc[formation_main,'MOM_var12'].copy();var_main.index=main_oos.index
training_for_target=master.loc['1995-02-28':'2004-12-31','MOM'].dropna()
assert len(training_for_target)==119
target_vol_primary=float(training_for_target.std(ddof=1))
print('1995–2004 MOM target monthly sigma:',target_vol_primary,'annualized',target_vol_primary*np.sqrt(12))
primary_weights=pd.DataFrame(index=main_oos.index)
primary_weights['Static 1x']=1.
primary_weights['Constant 1.6x']=1.6
for cap in (1,2):
    primary_weights[f'Vol-scale cap{cap}']=(target_vol_primary/vol_main).clip(lower=0,upper=cap)
    primary_weights[f'DM (1966) long cap{cap}']=(main_oos['DM (1966 train)']/var_main).clip(lower=0,upper=cap)
    primary_weights[f'DM (1995) long cap{cap}']=(main_oos['DM (1995 train)']/var_main).clip(lower=0,upper=cap)
    primary_weights[f'Composite long cap{cap}']=(main_oos['Composite (1995 train)']/var_main).clip(lower=0,upper=cap)
main_strategy=primary_weights.mul(main_oos['Actual'],axis=0)
main_strategy.to_csv(os.path.join(CLEAN,'MAIN_strategy_returns.csv'))
primary_weights.to_csv(os.path.join(CLEAN,'MAIN_strategy_weights.csv'))
main_perf_rows=[]
for strategy in main_strategy:
    pp=perf_stats(main_strategy[strategy]);alpha,tstat,r2,nn=ff5_alpha(main_strategy[strategy],master)
    pp.update({'Strategy':strategy,'Mean_weight':primary_weights[strategy].mean(),
               'Share_cap2':(primary_weights[strategy]>=2-1e-10).mean(),
               'Share_at_zero':(primary_weights[strategy]<1e-10).mean(),
               'FF5_alpha_ann':alpha,'FF5_t_alpha':tstat})
    main_perf_rows.append(pp)
main_perf=pd.DataFrame(main_perf_rows).set_index('Strategy')
main_perf.to_csv(os.path.join(TAB,'MAIN_table_08_portfolios_2005_2026.csv'))
display(main_perf[['N','Mean_ann','Vol_ann','Sharpe','MaxDrawdown','Mean_weight','Share_cap2']].round(3))
# Ex-post vol matching to Static 1x (no vol forecasting claim).
static_vol=float(main_perf.loc['Static 1x','Vol_ann'])
match_rows=[]
for name in main_strategy.columns:
    if name in ['Static 1x','Constant 1.6x']:continue
    scalar=static_vol/main_perf.loc[name,'Vol_ann']
    adjusted=main_strategy[name]*scalar
    pp=perf_stats(adjusted)
    match_rows.append({'Strategy':name,'Vol_normalizer_ex_post':scalar,'Matched_vol_ann':pp['Vol_ann'],
                       'Matched_return_ann':pp['Mean_ann'],'Matched_Sharpe':pp['Sharpe'],
                       'Matched_MaxDD':pp['MaxDrawdown'],'Return_diff_to_static_ann':pp['Mean_ann']-main_perf.loc['Static 1x','Mean_ann']})
match_tab=pd.DataFrame(match_rows)
match_tab.to_csv(os.path.join(TAB,'MAIN_table_09_vol_matching.csv'),index=False)
display(match_tab.round(4))


## 19. Sharpe differences, cap1 tests, and endpoint robustness

The moving-block bootstrap resamples paired observations in circular six-month blocks. We report percentile confidence intervals and percentile tail fractions; the latter should not be overstated as a fully null-imposed bootstrap significance test. Holm adjustments apply to the prespecified family below. The 2005–2023 / 2005–2026 comparison tests sensitivity to the last 28 months.

In [ ]:
from statsmodels.stats.multitest import multipletests
compare_list=[('Vol-scale cap2','Static 1x'),('Vol-scale cap1','Static 1x'),
              ('DM (1966) long cap2','Static 1x'),('DM (1966) long cap1','Static 1x'),
              ('DM (1995) long cap2','Static 1x'),('DM (1995) long cap1','Static 1x'),
              ('Composite long cap2','Static 1x'),('Composite long cap1','Static 1x'),
              ('DM (1966) long cap2','Vol-scale cap2'),('DM (1966) long cap1','Vol-scale cap1'),
              ('DM (1966) long cap1','DM (1966) long cap2'),('Vol-scale cap1','Vol-scale cap2')]
inf_rows=[]
for j,(a,b) in enumerate(compare_list):
    diff,pval,ci,num=moving_block_bootstrap_sharpe_diff(main_strategy[a],main_strategy[b],block=6,reps=5000,seed=157+j)
    inf_rows.append({'Strategy':a,'Benchmark':b,'N':num,'Sharpe_diff':diff,'Fraction_boot_diff_le_zero':pval,
                     'CI_2.5':ci[0],'CI_97.5':ci[1],'CI_excludes_zero':bool(ci[0]>0 or ci[1]<0)})
inf_tab=pd.DataFrame(inf_rows)
inf_tab['Holm_adjusted_tail_fraction']=multipletests(inf_tab['Fraction_boot_diff_le_zero'],method='holm')[1]
inf_tab.to_csv(os.path.join(TAB,'MAIN_table_10_sharpe_inference.csv'),index=False)
display(inf_tab.round(4))
end_rows=[]
for end in ['2023-12-31','2026-04-30']:
    sub=main_strategy.loc[:end]
    for strategy in main_strategy.columns:
        p=perf_stats(sub[strategy])
        end_rows.append({'End':end[:7],'Strategy':strategy,'N':p['N'],
                         'Mean_ann':p['Mean_ann'],'Vol_ann':p['Vol_ann'],
                         'Sharpe':p['Sharpe'],'MaxDD':p['MaxDrawdown']})
end_tab=pd.DataFrame(end_rows)
end_tab.to_csv(os.path.join(TAB,'MAIN_endpoint_comparison.csv'),index=False)
display(end_tab[end_tab.Strategy.isin(['Static 1x','Vol-scale cap1','Vol-scale cap2','DM (1966) long cap1','DM (1966) long cap2'])].round(4))


## 20. Primary figures and supplementary sentiment panel

The six original figure positions remain in the paper; the out-of-sample wealth, drawdown, and Sharpe displays below now use 2005–2026. The existing historical sentiment evidence remains strictly supplementary and is not part of the primary market-observable strategy comparison.

In [ ]:
# Replace the OOS plots with 2005–2026 main-sample figures (same figure numbers).
wealth=(1+main_strategy[['Static 1x','Constant 1.6x','Vol-scale cap2','Vol-scale cap1','DM (1966) long cap2','DM (1966) long cap1']]).cumprod()
plt.figure(figsize=(10,4.6))
((1+main_strategy['Static 1x']).cumprod()/((1+main_strategy['Static 1x']).cumprod().cummax())-1).mul(100).plot()
plt.title('Momentum drawdown, January 2005–April 2026');plt.ylabel('Drawdown (%)');plt.xlabel('Return month')
savefig(os.path.join(FIG,'figure_02_oos_drawdown.png'))
plt.figure(figsize=(10,5.6))
wealth.plot(ax=plt.gca(),lw=1.5)
plt.title('Momentum timing and leverage-only benchmark, 2005–2026')
plt.ylabel('Gross factor-growth index (start = 1)');plt.xlabel('Return month');plt.grid(alpha=.2)
savefig(os.path.join(FIG,'figure_05_oos_cumulative_strategy.png'))
plt.figure(figsize=(10,4.7))
main_perf.loc[['Static 1x','Constant 1.6x','Vol-scale cap2','Vol-scale cap1','DM (1966) long cap2','DM (1966) long cap1','Composite long cap2','Composite long cap1'],'Sharpe'].plot(kind='bar')
plt.ylabel('Annualized gross factor Sharpe');plt.title('Leverage caps and gross Sharpe, 2005–2026');plt.xticks(rotation=35,ha='right')
savefig(os.path.join(FIG,'figure_06_oos_sharpe_ratios.png'))
# Separate revised observation-window sentiment statistics: old 2005–2023 table remains available.
metricstab.to_csv(os.path.join(TAB,'APPENDIX_sentiment_2005_2023_metrics.csv'),index=False)
print('Main sample',main_oos.index.min(),main_oos.index.max(),'N=',len(main_oos))
print('Main output tables saved; principal companion outputs prefixed MAIN_.')


## 21. In-sample market-state tables corresponding to Tables 4 and 5

This section regenerates the 1966–2026, 1995–2026, and 2014–2026 reduced-form in-sample regressions without shortening DM's early eligible history.

In [ ]:
regression_rows=[]
for label,begin,end in [('Long 1966–2026','1966-01-31','2026-03-31'),
                        ('VIX 1995–2026','1995-01-31','2026-03-31'),
                        ('Recent 2014–2026','2014-01-31','2026-03-31')]:
    for model,xs in [('DM',['Bear','RV_ann','Bear_RV']),('Composite',['Composite','RV_ann'])]:
        if label.startswith('Long') and model=='Composite':continue
        sample=master.loc[begin:end].dropna(subset=['MOM_lead']+xs)
        mod=sm.OLS(sample['MOM_lead'],sm.add_constant(sample[xs])).fit(cov_type='HAC',cov_kwds={'maxlags':6})
        row={'Period':label,'Model':model,'N':len(sample),'Adj_R2':mod.rsquared_adj}
        for variable in ['Bear','RV_ann','Bear_RV','Composite']:
            row[variable]=mod.params.get(variable,np.nan)
            row['t_'+variable]=mod.tvalues.get(variable,np.nan)
        regression_rows.append(row)
market_regtab=pd.DataFrame(regression_rows)
market_regtab.to_csv(os.path.join(TAB,'MAIN_insample_market.csv'),index=False)
display(market_regtab.round(4))


## 22. Crash ranking: leave-one-event-out, ex-ante threshold sensitivity, and simple benchmarks

All crash cutoffs use the **1995–2004** formation-period MOM distribution. Predictor scores are observed at the previous month-end. Removing one event at a time is a sensitivity calculation, not a new out-of-sample test. The 95% intervals use circular block-bootstrap resampling of *paired outcomes and scores*; single-event samples with no crashes are excluded.


In [ ]:
from scipy.stats import rankdata

def final_auc(y, x):
    y=np.asarray(y,dtype=bool);x=np.asarray(x,dtype=float)
    valid=np.isfinite(x);y=y[valid];x=x[valid]
    k=int(y.sum());n0=int((~y).sum())
    return (rankdata(x)[y].sum()-k*(k+1)/2)/(k*n0) if k and n0 else np.nan

def final_block_auc_ci(y, score, block=6, reps=3000, seed=103):
    y=np.asarray(y,dtype=bool);score=np.asarray(score,dtype=float);n=len(y)
    rng=np.random.default_rng(seed);est=[]
    for _ in range(reps):
        starts=rng.integers(0,n,size=int(np.ceil(n/block)))
        ids=((starts[:,None]+np.arange(block))%n).ravel()[:n]
        val=final_auc(y[ids],score[ids])
        if np.isfinite(val):est.append(val)
    return np.quantile(est,[.025,.975]) if est else [np.nan,np.nan]

hist_mom=master.loc['1995-02-28':'2004-12-31','MOM'].dropna()
fixed_cutoff=float(hist_mom.quantile(.05))
main_r=main_oos['Actual']
crash05=(main_r<=fixed_cutoff)
dm_risk=-main_oos['DM (1966 train)']
final_loo=[]
for event_date in main_r[crash05].index:
    keep=main_r.index!=event_date
    final_loo.append({'Omitted_month':str(event_date.date()),'MOM_return':main_r.loc[event_date],
                      'AUC_remaining':final_auc(crash05[keep],dm_risk[keep])})
final_loo=pd.DataFrame(final_loo)
final_loo.to_csv(os.path.join(TAB,'FINAL_A6_leave_one_crash_out.csv'),index=False)

final_threshold=[]
for q in [.025,.05,.075,.10]:
    cutoff=float(hist_mom.quantile(q)); event=main_r<=cutoff
    ci=final_block_auc_ci(event,dm_risk,seed=109)
    final_threshold.append({'Training_quantile':q,'Cutoff':cutoff,'Crash_months':int(event.sum()),
                            'AUC':final_auc(event,dm_risk),'CI_low':ci[0],'CI_high':ci[1]})
final_threshold=pd.DataFrame(final_threshold)
final_threshold.to_csv(os.path.join(TAB,'FINAL_A7_crash_threshold_sensitivity.csv'),index=False)

form=main_oos.index-pd.offsets.MonthEnd(1)
final_scores={'DM (1966)':dm_risk}
for name,col,sgn in [('Prior MOM month','MOM',-1),('Trailing MOM volatility','MOM_vol12',1),
                     ('Market realized variance','RV_ann',1)]:
    series=master.reindex(form)[col];series.index=main_oos.index
    final_scores[name]=sgn*series
past12=master['MOM'].rolling(12,min_periods=12).sum().reindex(form)
past12.index=main_oos.index
final_scores['Prior 12-month MOM']=-past12
final_bench=[]
for name,s in final_scores.items():
    ci=final_block_auc_ci(crash05,s,seed=151)
    final_bench.append({'Score':name,'Fixed_cutoff_AUC':final_auc(crash05,s),
                        'CI_low':ci[0],'CI_high':ci[1]})
final_bench=pd.DataFrame(final_bench)
final_bench.to_csv(os.path.join(TAB,'FINAL_A8_simple_predictor_benchmarks.csv'),index=False)
final_blocks=[]
for block in [3,6,12]:
    ci=final_block_auc_ci(crash05,dm_risk,block=block,seed=103+block)
    final_blocks.append({'Block_length_months':block,'AUC':final_auc(crash05,dm_risk),
                         'CI_low':ci[0],'CI_high':ci[1]})
final_blocks=pd.DataFrame(final_blocks)
final_blocks.to_csv(os.path.join(TAB,'FINAL_A9_block_length_sensitivity.csv'),index=False)

print('Leave-one-event-out AUC');display(final_loo.round(4))
print('Threshold sensitivity');display(final_threshold.round(4))
print('Simple signal AUC');display(final_bench.round(4))
print('Block-length sensitivity');display(final_blocks.round(4))

# Paired uncertainty for DM versus the simple market-RV ranking score.
auc_diff_draws=[]
auc_rng=np.random.default_rng(142)
auc_y=crash05.to_numpy(); auc_dm=np.asarray(dm_risk);auc_rv=np.asarray(final_scores['Market realized variance'])
for _ in range(5000):
    starts=auc_rng.integers(0,len(auc_y),size=int(np.ceil(len(auc_y)/6)))
    ids=((starts[:,None]+np.arange(6))%len(auc_y)).ravel()[:len(auc_y)]
    a=final_auc(auc_y[ids],auc_dm[ids]);b=final_auc(auc_y[ids],auc_rv[ids])
    if np.isfinite(a) and np.isfinite(b):auc_diff_draws.append(a-b)
final_auc_difference=pd.DataFrame([{'Comparison':'DM minus market realized variance',
     'Delta_AUC':final_auc(auc_y,auc_dm)-final_auc(auc_y,auc_rv),
     'CI_low':np.quantile(auc_diff_draws,.025),'CI_high':np.quantile(auc_diff_draws,.975)}])
final_auc_difference.to_csv(os.path.join(TAB,'FINAL_AUC_DM_minus_market_RV.csv'),index=False)
display(final_auc_difference.round(4))


## 23. Economic value: crash-warning overlays and ex-ante volatility budgets

A DM warning means that the recursive forecast of the next MOM return is below zero. A zero-exposure rule sits out that month; the half-position rule halves the unit position. We include a constant-participation control with the **same average exposure** as the zero-exposure warning rule. The 15% annualized risk budget is set in advance and scaled each month using the **prior 12 realized monthly MOM returns**. Both budget-only and DM-overlay rules have the same 2× cap and the same budget. Realized volatility is **not** known or forced to equal 15%.


In [ ]:
warning=(main_oos['DM (1966 train)']<0)
warning_weight=pd.DataFrame(index=main_oos.index)
warning_weight['Static MOM']=1.
warning_weight['Constant participation']=1-warning.mean()
warning_weight['DM warning (0x)']=(~warning).astype(float)
warning_weight['DM warning (0.5x)']=1-.5*warning.astype(float)
lagged_vol=master.reindex(form)['MOM_vol12'];lagged_vol.index=main_oos.index
vol_budget_annual=.15
budget_w=(vol_budget_annual/np.sqrt(12)/lagged_vol).clip(0,2)
warning_weight['15% risk target only']=budget_w
warning_weight['15% target + DM warning']=budget_w*(~warning)
warning_weight['15% target + half warning']=budget_w*(1-.5*warning)
warning_weight['15% target + DM continuous']=np.minimum(budget_w,primary_weights['DM (1966) long cap2'])
warning_return=warning_weight.mul(main_r,axis=0)
final_policy_rows=[]
for name in warning_return.columns:
    p=perf_stats(warning_return[name])
    final_policy_rows.append({'Strategy':name,'N':p['N'],'Return_pa':p['Mean_ann'],
        'Vol_pa':p['Vol_ann'],'Sharpe':p['Sharpe'],'Max_drawdown':p['MaxDrawdown'],
        'Mean_weight':warning_weight[name].mean(),'Cumulative_return':p['CumReturn']})
final_policy=pd.DataFrame(final_policy_rows)
final_policy.to_csv(os.path.join(TAB,'FINAL_table_11_warning_risk_budget.csv'),index=False)
warning_weight.to_csv(os.path.join(CLEAN,'FINAL_warning_risk_target_weights.csv'))
warning_return.to_csv(os.path.join(CLEAN,'FINAL_warning_risk_target_returns.csv'))

final_warning_groups=[]
for label,mask in [('Correct warnings',warning & crash05),('False alarms',warning & ~crash05),
                    ('Unwarned crashes',~warning & crash05)]:
    final_warning_groups.append({'Group':label,'Months':int(mask.sum()),
      'Sum_of_static_MOM_returns':float(main_r[mask].sum()),
      'Sum_of_risk_target_returns':float((main_r*budget_w)[mask].sum())})
final_warning_groups=pd.DataFrame(final_warning_groups)
final_warning_groups.to_csv(os.path.join(TAB,'FINAL_warning_hit_false_alarm_attribution.csv'),index=False)
final_budget_sensitivity=[]
for budget in [.12,.15,.20]:
    v=(budget/np.sqrt(12)/lagged_vol).clip(0,2)
    for name,w in [('Volatility target only',v),('Volatility target + DM warning',v*(~warning))]:
        pp=perf_stats(w*main_r)
        final_budget_sensitivity.append({'Budget_ann':budget,'Strategy':name,
                'Realized_vol_pa':pp['Vol_ann'],'Sharpe':pp['Sharpe'],'Max_drawdown':pp['MaxDrawdown']})
final_budget_sensitivity=pd.DataFrame(final_budget_sensitivity)
final_budget_sensitivity.to_csv(os.path.join(TAB,'FINAL_risk_budget_sensitivity.csv'),index=False)
display(final_policy.round(4));display(final_warning_groups.round(4));display(final_budget_sensitivity.round(4))


## 24. Paired bootstrap uncertainty for warning overlays

These are percentile intervals, not null-imposed p-values. Each resample retains the matching return months for both strategies. The timing strategies are not treated as established improvements if the interval contains zero.


In [ ]:
final_compare=[('DM warning (0x)','Static MOM'),
               ('DM warning (0x)','Constant participation'),
               ('15% target + DM warning','15% risk target only'),
               ('15% target + DM continuous','15% risk target only')]
final_inference=[]
for j,(a,b) in enumerate(final_compare):
    diff,tail,ci,nn=moving_block_bootstrap_sharpe_diff(
        warning_return[a],warning_return[b],block=6,reps=5000,seed=1550+j)
    final_inference.append({'Strategy':a,'Benchmark':b,'N':nn,'Sharpe_diff':diff,
                            'CI_low':ci[0],'CI_high':ci[1],'Bootstrap_fraction_le_zero':tail})
final_inference=pd.DataFrame(final_inference)
final_inference.to_csv(os.path.join(TAB,'FINAL_warning_sharpe_inference.csv'),index=False)
display(final_inference.round(4))
